# Activity: logistic regression gradients

**Goal:** connect the chain rule for one observation to a vectorized full-batch gradient and one gradient-descent step. 

Complete the algebra in the Markdown cells and replace each `None` marked `TODO` in the code cells. Run the notebook from top to bottom as you go. The checks help locate mistakes, but your written explanations are part of the activity.

We use the **sum** of negative log-likelihoods, keep the intercept $b$ separate from $X$, and use no regularization.


## 1. Trace the derivatives for one observation

For one feature vector $x_i\in\mathbb R^d$ and label $y_i\in\{0,1\}$,

$$
z_i=w^\top x_i+b,\qquad p_i=\sigma(z_i)=\frac{1}{1+e^{-z_i}},
$$

$$
J_i=-\left[y_i\log p_i+(1-y_i)\log(1-p_i)\right].
$$

The chain we need is $w,b\longrightarrow z_i\longrightarrow p_i\longrightarrow J_i$. First derive the middle two links; then connect them to the parameters.

**A. Sigmoid derivative.** Differentiate $(1+e^{-z_i})^{-1}$. The first simplification is shown. Rewrite it using $p_i$ and $1-p_i$:

$$
\frac{dp_i}{dz_i}
=\frac{e^{-z_i}}{(1+e^{-z_i})^2}
=\boxed{\underline{\hspace{1.8in}}}.
$$

Which factor in the fraction equals $p_i$? Which equals $1-p_i$?

> **Your explanation:**


**B. Loss derivative.** Hold $y_i$ fixed. Differentiate each logarithm with respect to $p_i$; remember the inner derivative of $1-p_i$.

$$
\frac{dJ_i}{dp_i}
=\frac{d}{dp_i}\left[-y_i\log p_i-(1-y_i)\log(1-p_i)\right]
=\boxed{\underline{\hspace{3.0in}}}.
$$

> **Show the two terms and their signs:**

**C. Chain rule.** Multiply your answers from A and B, cancel factors, and simplify. Fill in **both** missing lines:

$$
\begin{aligned}
\frac{dJ_i}{dz_i}
&=\frac{dJ_i}{dp_i}\frac{dp_i}{dz_i}\\
&=\underline{\hspace{3.4in}}\\[12pt]
&=-y_i(1-p_i)+(1-y_i)p_i\\
&=\boxed{\underline{\hspace{1.2in}}}.
\end{aligned}
$$

In a sentence, what does the last expression compare?

> **Your explanation:**


**D. Reach the weights and intercept.** Expand the score as $z_i=\sum_{j=1}^d w_jx_{ij}+b$. Fill in the local derivatives, then apply the chain rule from C.

$$
\frac{\partial z_i}{\partial w_j}=\underline{\hspace{0.8in}},
\qquad
\frac{\partial z_i}{\partial b}=\underline{\hspace{0.8in}}.
$$

$$
\frac{\partial J_i}{\partial w_j}
=\frac{dJ_i}{dz_i}\frac{\partial z_i}{\partial w_j}
=\underline{\hspace{1.6in}},
\qquad
\frac{\partial J_i}{\partial b}=\underline{\hspace{1.3in}}.
$$

Collecting the $d$ weight derivatives gives the **one-point vector gradient**:

$$
\nabla_w J_i=\underline{\hspace{2.0in}}\quad\text{(a vector of length $d$)}.
$$


### Put numbers on the first point

Use $x_1=(-1)$, $y_1=0$, $w^{(0)}=(0)$, and $b^{(0)}=0$. Compute the score, probability, $dJ_1/dz_1$, and both one-point gradients. Use your symbolic answers above rather than guessing from the label.


In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

x_i = np.array([-1.0])
y_i = 0.0
w0 = np.array([0.0])
b0 = 0.0

z_i = None        # TODO: w0 @ x_i + b0
p_i = None        # TODO: apply sigmoid to z_i
dJ_dz_i = None    # TODO: simplify the chain-rule result
grad_w_i = None   # TODO: weight gradient for this one point; shape (1,)
grad_b_i = None   # TODO: intercept gradient for this one point

assert all(value is not None for value in
           (z_i, p_i, dJ_dz_i, grad_w_i, grad_b_i)), \
    "Replace every None in this cell before continuing."
assert np.shape(grad_w_i) == (1,), "Keep the weight gradient as a length-1 array."
print(f"z_1 = {z_i:.3f}; p_1 = {p_i:.3f}; dJ_1/dz_1 = {dJ_dz_i:.3f}")
print(f"one-point weight gradient = {grad_w_i}; intercept gradient = {grad_b_i:.3f}")


**Pause and predict:** If you took a small step using only this point, would the weight increase or decrease? How would that change its score, given $x_1=-1$? Explain why the direction fits $y_1=0$.

> **Your explanation:**


## 2. Assemble a full-batch gradient

Now add two observations to the point you just analyzed. The full training set is:

| $i$ | $x_i$ | $y_i$ |
|---:|---:|---:|
| 1 | $-1$ | $0$ |
| 2 | $0$ | $0$ |
| 3 | $1$ | $1$ |

We will build $X\in\mathbb R^{n\times d}$ with **one observation per row**, $y\in\mathbb R^n$, and $w\in\mathbb R^d$. Here $n=3$ and $d=1$. NumPy stores $y$ and $w$ as one-dimensional arrays; the mathematical expressions have the same dimensions as column vectors.


In [ ]:
X = np.array([[-1.0], [0.0], [1.0]])
y = np.array([0.0, 0.0, 1.0])
n, d = X.shape

print("X shape:", X.shape, "y shape:", y.shape, "w shape:", w0.shape)
assert X.shape == (3, 1) and y.shape == (3,) and w0.shape == (1,)


At the **same current** $(w^{(0)},b^{(0)})$, calculate every score and probability. Let $r_i=p_i-y_i$ and collect the three $r_i$ values in `residual`. The contribution from row $i$ to the weight gradient is $r_i x_i$; the contribution to the intercept gradient is $r_i$.

**Before coding:** Predict the sign of $r_i$ for each of the three points. Why does the middle point affect the intercept gradient but not the weight gradient?

> **Your prediction:**

Hint for the code: `residual[:, None]` has shape $(n,1)$, so multiplying it by `X` makes one weight-gradient row per observation.


In [ ]:
scores = None           # TODO: all z_i from X, w0, and b0; shape (n,)
probs = None            # TODO: apply sigmoid to scores
residual = None         # TODO: each p_i - y_i
row_grad_w = None       # TODO: one r_i * x_i per row; shape (n, d)
row_grad_b = None       # TODO: one r_i per row; shape (n,)

assert all(value is not None for value in
           (scores, probs, residual, row_grad_w, row_grad_b)), \
    "Replace every None in this cell before continuing."
assert np.shape(scores) == np.shape(probs) == np.shape(residual) == (n,)
assert np.shape(row_grad_w) == (n, d) and np.shape(row_grad_b) == (n,)

print(" i     z_i      p_i      r_i    r_i*x_i")
for i in range(n):
    print(f"{i+1:2d}  {scores[i]:7.3f}  {probs[i]:7.3f}  "
          f"{residual[i]:7.3f}  {row_grad_w[i, 0]:9.3f}")


**Check row 1:** Compare `scores[0]`, `probs[0]`, `row_grad_w[0]`, and `row_grad_b[0]` with your one-point results. They should match because the parameters have not changed.

> **Your comparison:**

For the **sum** loss $J(w,b)=\sum_{i=1}^n J_i$, add the row contributions:

$$
\nabla_w J=\sum_{i=1}^n r_i x_i,
\qquad
\frac{\partial J}{\partial b}=\sum_{i=1}^n r_i.
$$

Write the two numerical sums for these three points, including the signs, before running the next cell.

> **Weight sum:**
>
> **Intercept sum:**


In [ ]:
# This route adds the per-point contributions you just computed.
grad_w_from_rows = row_grad_w.sum(axis=0)
grad_b_from_rows = float(row_grad_b.sum())

print("sum of row weight gradients:", grad_w_from_rows)
print("sum of row intercept gradients:", grad_b_from_rows)


### Turn the sums into matrix multiplication

The additive gradient is already enough for gradient descent. Matrix multiplication gives a compact way to calculate **the same sums for every weight at once**. Define the residual vector

$$
r=p-y=
\begin{bmatrix}
p_1-y_1\\
\vdots\\
p_n-y_n
\end{bmatrix}.
$$

Because row $i$ of $X$ is $x_i^\top$, its transpose has the feature vectors as columns:

$$
X^\top r
=
\begin{bmatrix}x_1 & x_2 & \cdots & x_n\end{bmatrix}
\begin{bmatrix}r_1\\r_2\\\vdots\\r_n\end{bmatrix}
=
\sum_{i=1}^n r_i x_i.
$$

For our one-feature dataset, this reads

$$
X^\top r
=
\begin{bmatrix}-1 & 0 & 1\end{bmatrix}
\begin{bmatrix}r_1\\r_2\\r_3\end{bmatrix}
=(-1)r_1+0r_2+1r_3.
$$

The intercept multiplies every observation by $1$, so its sum is

$$
\mathbf 1^\top r
=
\begin{bmatrix}1 & 1 & 1\end{bmatrix}
\begin{bmatrix}r_1\\r_2\\r_3\end{bmatrix}
=r_1+r_2+r_3.
$$

Thus the **vectorized gradients for the sum NLL** are

$$
\boxed{\nabla_w J=X^\top(p-y)},
\qquad
\boxed{\frac{\partial J}{\partial b}=\mathbf 1^\top(p-y)}.
$$

The shapes explain the result: $X^\top$ is $d\times n$, $p-y$ is $n\times1$, so $X^\top(p-y)$ is $d\times1$. The intercept expression is $1\times n$ times $n\times1$, giving one scalar. In NumPy, we store $p-y$ and $w$ as one-dimensional arrays, so the weight-gradient array has shape $(d,)$.

**Check your understanding:** Match the three terms in $(-1)r_1+0r_2+1r_3$ to the weight-gradient rows you printed. Why does the intercept use a vector of ones?

> **Your explanation:**


In [ ]:
grad_w = None        # TODO: vectorized weight gradient using X.T and residual
grad_b = None        # TODO: vectorized intercept gradient using residual

assert grad_w is not None and grad_b is not None, \
    "Replace both None values with the vectorized gradients."
assert np.shape(grad_w) == (d,), "The weight gradient must have one entry per feature."
assert np.ndim(grad_b) == 0, "The intercept gradient must be a scalar."
np.testing.assert_allclose(grad_w, grad_w_from_rows)
np.testing.assert_allclose(grad_b, grad_b_from_rows)
np.testing.assert_allclose(scores[0], z_i)
np.testing.assert_allclose(probs[0], p_i)
np.testing.assert_allclose(residual[0], dJ_dz_i)
np.testing.assert_allclose(row_grad_w[0], grad_w_i)
np.testing.assert_allclose(row_grad_b[0], grad_b_i)
print("Both gradient routes agree:", grad_w, grad_b)


## 3. Take one full-batch gradient-descent step

Use learning rate $\alpha=\tfrac12$. Write the simultaneous update first:

$$
w^{(1)}=\underline{\hspace{2.8in}},
\qquad
b^{(1)}=\underline{\hspace{2.8in}}.
$$

Both right-hand sides must use the gradients evaluated at $(w^{(0)},b^{(0)})$. Predict the direction of each change before coding.

> **Weight direction:**
>
> **Intercept direction:**


In [ ]:
alpha = 0.5
w1 = None           # TODO: one full-batch weight update from w0
b1 = None           # TODO: one full-batch intercept update from b0

assert w1 is not None and b1 is not None, "Replace both None values with the updates."
assert np.shape(w1) == (d,) and np.ndim(b1) == 0
print("updated weight:", w1)
print("updated intercept:", b1)


We can check the step with the same **sum** loss. For a score $z$, the binary NLL can be written as $\log(1+e^z)-yz$; `np.logaddexp(0, z)` evaluates the first term stably. The next cell is provided so you can focus on the gradient and update.


In [ ]:
def sum_nll(X, y, w, b):
    z = X @ w + b
    return float(np.sum(np.logaddexp(0, z) - y * z))

loss_before = sum_nll(X, y, w0, b0)
loss_after = sum_nll(X, y, w1, b1)
new_probs = sigmoid(X @ w1 + b1)

print(f"sum NLL before: {loss_before:.4f}; after: {loss_after:.4f}")
print("probabilities after the update:", np.round(new_probs, 4))
assert loss_after < loss_before, "Recheck the gradient signs and update direction."


## 4. Explain what happened

1. Report your numerical $\nabla_w J$, $\partial J/\partial b$, $w^{(1)}$, and $b^{(1)}$. Show the arithmetic for the update, not just the printed values.
2. Compare the probabilities before and after the update for the negative point at $x=-1$ and positive point at $x=1$. Do they move in the expected directions?
3. We used the **sum** NLL. If the objective were the **mean** NLL $J/n$, how would each gradient and this one-step update change if $\alpha$ stayed $0.5$?
4. If every row had **two** features, what would the shapes of $X$, $w$, and $\nabla_w J$ be? Would the vectorized formula itself change?

> **Your responses:**
